In [1]:
import os
import json
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, average_precision_score
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.6 MB/s eta 0:00:00


In [4]:
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

경로 설정 및 파일들 불러오기

In [5]:
base_dir = '/content/drive/MyDrive/itda'

mlp_dir = base_dir + '/baseline model_mlp'
graphsage_dir = base_dir + '/baseline model_graphSAGE'

os.makedirs(graphsage_dir, exist_ok=True)

node_feature_path = mlp_dir + '/itda_node_features.csv'
mlp_result_path = mlp_dir + '/itda_mlp_result.csv'
config_path = mlp_dir + '/itda_experiment_config.json'

print("node feature file exists:", os.path.exists(node_feature_path))
print("mlp result file exists:", os.path.exists(mlp_result_path))
print("config file exists:", os.path.exists(config_path))

node feature file exists: True
mlp result file exists: True
config file exists: True


In [7]:
df = pd.read_csv(node_feature_path)
df['date'] = pd.to_datetime(df['date'])
print(df.head())


   user_id  prod_id  rating  label       date  \
0     8253     3237     5.0      0 2005-08-13   
1    17480     3237     3.0      0 2005-09-19   
2   149466     3237     5.0      0 2005-10-11   
3    41746     3237     5.0      0 2005-10-27   
4   192388     3237     5.0      1 2005-10-28   

                                                text  original_review_id  \
0  One of the original pizza places. Worth the lo...              378701   
1  Expect a wait before your first bite into this...              378700   
2  Could this pizza be anymore delicious?  The wh...              378699   
3  Shocker!  Everyone thinks that this is best pi...              378698   
4  One of my favorite places in the city.  The pi...              374686   

   review_id  split  train_mask  valid_mask  test_mask  text_length  \
0          0  train        True       False      False          202   
1          1   test       False       False       True          303   
2          2  train        True    

In [8]:
df = df.sort_values('review_id').reset_index(drop=True)

if not np.array_equal(df['review_id'].values, np.arange(len(df))):
    print("review_id가 0부터 연속된 번호가 아닙니다. 다시 생성합니다.")
    df['review_id'] = np.arange(len(df))
else:
    print("review_id 정상:", df['review_id'].min(), "~", df['review_id'].max())

review_id 정상: 0 ~ 14999


In [9]:
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = json.load(f)
    feature_cols = config['feature_cols']
else:
    feature_cols = [
        'rating',
        'text_length',
        'word_count',
        'user_review_count',
        'prod_review_count',
        'days_from_start'
    ]

print("사용 feature:")
print(feature_cols)

print(df[feature_cols].head())

사용 feature:
['rating', 'text_length', 'word_count', 'user_review_count', 'prod_review_count', 'days_from_start']
   rating  text_length  word_count  user_review_count  prod_review_count  \
0     5.0          202          31                  2                990   
1     3.0          303          57                  1                990   
2     5.0          202          32                  1                990   
3     5.0          287          50                  1                990   
4     5.0          393          76                  1                990   

   days_from_start  
0                0  
1               37  
2               59  
3               75  
4               76  


R-U-R edge 생성 (undirected)

In [13]:
def build_rur_pairs(df):
    pair_blocks = []

    for user_id, group in df.groupby('user_id'):
        nodes = group['review_id'].values
        n = len(nodes)

        if n < 2:
            continue

        row_idx, col_idx = np.triu_indices(n, k=1)

        src = nodes[row_idx]
        dst = nodes[col_idx]

        a = np.minimum(src, dst)
        b = np.maximum(src, dst)

        pairs = np.vstack([a, b]).T
        pair_blocks.append(pairs)

    if len(pair_blocks) == 0:
        return pd.DataFrame(columns=['a', 'b'])

    pair_array = np.concatenate(pair_blocks, axis=0)
    pair_df = pd.DataFrame(pair_array, columns=['a', 'b'])
    pair_df = pair_df.drop_duplicates().reset_index(drop=True)

    return pair_df


rur_pair_df = build_rur_pairs(df)

print("R-U-R undirected pair count:", len(rur_pair_df))
rur_pair_df.head()

R-U-R undirected pair count: 1434


,a,b
0,5394,11890
1,5,1096
2,6812,12674
3,5588,13541
4,4368,10582


Product-Time edge 생성

In [14]:
# 같은 prod_id에 대해 30일 이내 작성된 review끼리 연결
def build_product_time_pairs(df, window_days=30):
    pair_blocks = []
    window = np.timedelta64(window_days, 'D')

    for prod_id, group in df.groupby('prod_id'):
        group = group.sort_values('date')

        nodes = group['review_id'].values
        dates = group['date'].values.astype('datetime64[D]')
        n = len(nodes)

        if n < 2:
            continue

        left = 0

        for right in range(n):
            while dates[right] - dates[left] > window:
                left += 1

            if right > left:
                prev_nodes = nodes[left:right]
                current_node = nodes[right]

                a = np.minimum(current_node, prev_nodes)
                b = np.maximum(current_node, prev_nodes)

                pairs = np.vstack([a, b]).T
                pair_blocks.append(pairs)

    if len(pair_blocks) == 0:
        return pd.DataFrame(columns=['a', 'b'])

    pair_array = np.concatenate(pair_blocks, axis=0)
    pair_df = pd.DataFrame(pair_array, columns=['a', 'b'])
    pair_df = pair_df.drop_duplicates().reset_index(drop=True)

    return pair_df


product_time_pair_df = build_product_time_pairs(df, window_days=30)

print("Product-Time undirected pair count:", len(product_time_pair_df))
product_time_pair_df.head()

Product-Time undirected pair count: 1486705


,a,b
0,1,2
1,2,3
2,2,4
3,3,4
4,5,6


edge 합치기; R-U-R edge와 Product-Time edge를 하나의 edge_index로 합친다. GraphSAGE baseline에서는 relation type을 따로 구분하지 않고 하나로 합쳐서 쓰기 떄문 + 중복되는 엣지 있다면 제거

In [15]:
rur_pair_df['edge_type'] = 'R-U-R'
product_time_pair_df['edge_type'] = 'Product-Time'

combined_pair_df = pd.concat(
    [rur_pair_df, product_time_pair_df],
    axis=0
).reset_index(drop=True)

print("중복 제거 전 undirected pair count:", len(combined_pair_df))

# relation type은 제거하고, node pair 기준으로 중복 제거
unique_pair_df = combined_pair_df[['a', 'b']].drop_duplicates().reset_index(drop=True)

print("중복 제거 후 undirected pair count:", len(unique_pair_df))
print("제거된 중복 pair 수:", len(combined_pair_df) - len(unique_pair_df))

unique_pair_df.head()

중복 제거 전 undirected pair count: 1488139
중복 제거 후 undirected pair count: 1488139
제거된 중복 pair 수: 0


,a,b
0,5394,11890
1,5,1096
2,6812,12674
3,5588,13541
4,4368,10582


PyG용 edge_index 생성

In [17]:
#PyTorch Geometric에서는 무방향 edge도 내부적으로는 양방향으로 넣음
src_forward = unique_pair_df['a'].values
dst_forward = unique_pair_df['b'].values

src_backward = unique_pair_df['b'].values
dst_backward = unique_pair_df['a'].values

src = np.concatenate([src_forward, src_backward])
dst = np.concatenate([dst_forward, dst_backward])

edge_index_np = np.vstack([src, dst])

edge_index = torch.tensor(edge_index_np, dtype=torch.long)

print("undirected pair count:", len(unique_pair_df))
print("directed edge_index shape:", edge_index.shape)
print("directed edge count:", edge_index.shape[1])

undirected pair count: 1488139
directed edge_index shape: torch.Size([2, 2976278])
directed edge count: 2976278


edge 관련 파일 저장

In [18]:
edge_index_path = graphsage_dir + '/itda_combined_edge_index.pt'
torch.save(edge_index, edge_index_path)

edge_summary = pd.DataFrame({
    'item': [
        'R-U-R undirected pairs',
        'Product-Time undirected pairs',
        'Combined pairs before dedup',
        'Combined pairs after dedup',
        'Overlap pairs between relations',
        'Final directed edge count for PyG'
    ],
    'count': [
        len(rur_unique),
        len(pt_unique),
        len(combined_pair_df),
        len(unique_pair_df),
        len(overlap_pair_df),
        edge_index.shape[1]
    ]
})

edge_summary_path = graphsage_dir + '/itda_edge_summary.csv'
edge_summary.to_csv(edge_summary_path, index=False, encoding='utf-8-sig')

print("edge_index 저장:", edge_index_path)
print("edge_summary 저장:", edge_summary_path)

edge_summary

edge_index 저장: /content/drive/MyDrive/itda/baseline model_graphSAGE/itda_combined_edge_index.pt
edge_summary 저장: /content/drive/MyDrive/itda/baseline model_graphSAGE/itda_edge_summary.csv


,item,count
0,R-U-R undirected pairs,1434
1,Product-Time undirected pairs,1486705
2,Combined pairs before dedup,1488139
3,Combined pairs after dedup,1488139
4,Overlap pairs between relations,0
5,Final directed edge count for PyG,2976278


GraphSAGE용 Data 객체 만들기

In [19]:
train_mask_np = (df['split'] == 'train').values
valid_mask_np = (df['split'] == 'valid').values
test_mask_np = (df['split'] == 'test').values

X = df[feature_cols].values
y = df['label'].values

scaler = StandardScaler()
scaler.fit(X[train_mask_np])

X_scaled = scaler.transform(X)

x_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
valid_mask = torch.tensor(valid_mask_np, dtype=torch.bool)
test_mask = torch.tensor(test_mask_np, dtype=torch.bool)

data = Data(
    x=x_tensor,
    edge_index=edge_index,
    y=y_tensor,
    train_mask=train_mask,
    valid_mask=valid_mask,
    test_mask=test_mask
)

print(data)
print("num_nodes:", data.num_nodes)
print("num_edges:", data.num_edges)
print("num_features:", data.num_features)

Data(x=[15000, 6], edge_index=[2, 2976278], y=[15000], train_mask=[15000], valid_mask=[15000], test_mask=[15000])
num_nodes: 15000
num_edges: 2976278
num_features: 6


GraphSAGE 모델 정의

In [20]:
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, dropout=0.3):
        super(GraphSAGE, self).__init__()

        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.linear = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.linear(x).view(-1)
        return x

학습 설정

In [21]:
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

data = data.to(device)

input_dim = data.num_features
model = GraphSAGE(input_dim=input_dim, hidden_dim=32, dropout=0.3).to(device)

train_y = data.y[data.train_mask]
positive_count = train_y.sum().item()
negative_count = len(train_y) - positive_count

if positive_count == 0:
    raise ValueError("train set에 label=1인 데이터가 없습니다.")

pos_weight_value = negative_count / positive_count
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

print("input_dim:", input_dim)
print("positive_count:", positive_count)
print("negative_count:", negative_count)
print("pos_weight:", pos_weight_value)

device: cpu
input_dim: 6
positive_count: 1003.0
negative_count: 8597.0
pos_weight: 8.571286141575275


평가 함수

In [22]:
def evaluate_gnn(model, data, mask, threshold=0.5):
    model.eval()

    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        probs = torch.sigmoid(logits[mask]).detach().cpu().numpy()

    y_true = data.y[mask].detach().cpu().numpy()
    preds = (probs >= threshold).astype(int)

    acc = accuracy_score(y_true, preds)
    macro_f1 = f1_score(y_true, preds, average='macro', zero_division=0)
    pr_auc = average_precision_score(y_true, probs)

    return acc, macro_f1, pr_auc

GraphSAGE 학습

In [23]:
EPOCHS = 100

history = []

best_valid_pr_auc = -1
best_model_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()

    optimizer.zero_grad()

    logits = model(data.x, data.edge_index)
    loss = criterion(logits[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    train_acc, train_f1, train_pr_auc = evaluate_gnn(model, data, data.train_mask)
    valid_acc, valid_f1, valid_pr_auc = evaluate_gnn(model, data, data.valid_mask)

    history.append({
        'epoch': epoch,
        'loss': loss.item(),
        'train_acc': train_acc,
        'train_macro_f1': train_f1,
        'train_pr_auc': train_pr_auc,
        'valid_acc': valid_acc,
        'valid_macro_f1': valid_f1,
        'valid_pr_auc': valid_pr_auc
    })

    if valid_pr_auc > best_valid_pr_auc:
        best_valid_pr_auc = valid_pr_auc
        best_model_state = copy.deepcopy(model.state_dict())

    if epoch % 10 == 0:
        print(
            "epoch:", epoch,
            "loss:", round(loss.item(), 4),
            "valid_macro_f1:", round(valid_f1, 4),
            "valid_pr_auc:", round(valid_pr_auc, 4)
        )

epoch: 10 loss: 1.2192 valid_macro_f1: 0.3438 valid_pr_auc: 0.1972
epoch: 20 loss: 1.1949 valid_macro_f1: 0.465 valid_pr_auc: 0.1942
epoch: 30 loss: 1.1707 valid_macro_f1: 0.5003 valid_pr_auc: 0.1974
epoch: 40 loss: 1.1624 valid_macro_f1: 0.5109 valid_pr_auc: 0.2011
epoch: 50 loss: 1.1502 valid_macro_f1: 0.5182 valid_pr_auc: 0.2075
epoch: 60 loss: 1.137 valid_macro_f1: 0.5285 valid_pr_auc: 0.2138
epoch: 70 loss: 1.1304 valid_macro_f1: 0.5324 valid_pr_auc: 0.2179
epoch: 80 loss: 1.1255 valid_macro_f1: 0.5306 valid_pr_auc: 0.2222
epoch: 90 loss: 1.1342 valid_macro_f1: 0.5386 valid_pr_auc: 0.2252
epoch: 100 loss: 1.1198 valid_macro_f1: 0.5386 valid_pr_auc: 0.2269


valid set에서 best threshold 찾기

In [24]:
model.load_state_dict(best_model_state)
model.eval()

with torch.no_grad():
    logits = model(data.x, data.edge_index)
    valid_probs = torch.sigmoid(logits[data.valid_mask]).detach().cpu().numpy()

y_valid = data.y[data.valid_mask].detach().cpu().numpy()

best_threshold = 0
best_valid_macro_f1 = -1

for threshold in np.arange(0.05, 0.95, 0.01):
    valid_preds = (valid_probs >= threshold).astype(int)
    macro_f1 = f1_score(y_valid, valid_preds, average='macro', zero_division=0)

    if macro_f1 > best_valid_macro_f1:
        best_valid_macro_f1 = macro_f1
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best valid macro f1:", best_valid_macro_f1)
print("Best valid PR-AUC:", best_valid_pr_auc)

Best threshold: 0.6800000000000002
Best valid macro f1: 0.5918061025698186
Best valid PR-AUC: 0.2272599410567232


test 성능 평가

In [25]:
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    test_probs = torch.sigmoid(logits[data.test_mask]).detach().cpu().numpy()

y_test = data.y[data.test_mask].detach().cpu().numpy()
test_preds = (test_probs >= best_threshold).astype(int)

sage_test_acc = accuracy_score(y_test, test_preds)
sage_test_macro_f1 = f1_score(y_test, test_preds, average='macro', zero_division=0)
sage_test_pr_auc = average_precision_score(y_test, test_probs)

print("GraphSAGE Test Accuracy:", sage_test_acc)
print("GraphSAGE Test Macro F1:", sage_test_macro_f1)
print("GraphSAGE Test PR-AUC:", sage_test_pr_auc)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_preds))

print("\nClassification Report")
print(classification_report(y_test, test_preds, zero_division=0))

GraphSAGE Test Accuracy: 0.8446666666666667
GraphSAGE Test Macro F1: 0.6135559486952675
GraphSAGE Test PR-AUC: 0.26322364204070237

Confusion Matrix
[[2427  259]
 [ 207  107]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.92      0.90      0.91      2686
         1.0       0.29      0.34      0.31       314

    accuracy                           0.84      3000
   macro avg       0.61      0.62      0.61      3000
weighted avg       0.86      0.84      0.85      3000



저장

In [26]:
# 1.edge 정보 저장
# edge_index 저장
edge_index_path = graphsage_dir + '/itda_combined_edge_index.pt'
torch.save(edge_index, edge_index_path)

# edge 요약 저장
edge_summary = pd.DataFrame({
    'item': [
        'R-U-R undirected pairs',
        'Product-Time undirected pairs',
        'Combined pairs after dedup',
        'Final directed edge count for PyG'
    ],
    'count': [
        len(rur_pair_df[['a', 'b']].drop_duplicates()),
        len(product_time_pair_df[['a', 'b']].drop_duplicates()),
        len(unique_pair_df),
        data.num_edges
    ]
})

edge_summary_path = graphsage_dir + '/itda_edge_summary.csv'
edge_summary.to_csv(edge_summary_path, index=False, encoding='utf-8-sig')

print("edge_index 저장 완료:", edge_index_path)
print("edge_summary 저장 완료:", edge_summary_path)
edge_summary

edge_index 저장 완료: /content/drive/MyDrive/itda/baseline model_graphSAGE/itda_combined_edge_index.pt
edge_summary 저장 완료: /content/drive/MyDrive/itda/baseline model_graphSAGE/itda_edge_summary.csv


,item,count
0,R-U-R undirected pairs,1434
1,Product-Time undirected pairs,1486705
2,Combined pairs after dedup,1488139
3,Final directed edge count for PyG,2976278


In [28]:
#2. 학습 history 저장
history_df = pd.DataFrame(history)

history_path = graphsage_dir + '/itda_graphsage_history.csv'
history_df.to_csv(history_path, index=False, encoding='utf-8-sig')

print("GraphSAGE history 저장 완료:", history_path)

GraphSAGE history 저장 완료: /content/drive/MyDrive/itda/baseline model_graphSAGE/itda_graphsage_history.csv


In [29]:
# 3. 최종 결과 저장
cm = confusion_matrix(y_test, test_preds)

graphsage_result = pd.DataFrame({
    'model': ['GraphSAGE'],
    'test_accuracy': [sage_test_acc],
    'test_macro_f1': [sage_test_macro_f1],
    'test_pr_auc': [sage_test_pr_auc],
    'best_valid_pr_auc': [best_valid_pr_auc],
    'best_valid_macro_f1': [best_valid_macro_f1],
    'best_threshold': [best_threshold],

    # confusion matrix
    'true_negative': [cm[0, 0]],
    'false_positive': [cm[0, 1]],
    'false_negative': [cm[1, 0]],
    'true_positive': [cm[1, 1]],

    # edge info
    'undirected_pair_count': [len(unique_pair_df)],
    'directed_edge_count': [data.num_edges],

    'note': ['R-U-R + Product-Time 30 days, GraphSAGE baseline']
})

result_path = graphsage_dir + '/itda_graphsage_result.csv'
graphsage_result.to_csv(result_path, index=False, encoding='utf-8-sig')

print("GraphSAGE result 저장 완료:", result_path)
graphsage_result

GraphSAGE result 저장 완료: /content/drive/MyDrive/itda/baseline model_graphSAGE/itda_graphsage_result.csv


,model,test_accuracy,test_macro_f1,test_pr_auc,best_valid_pr_auc,best_valid_macro_f1,best_threshold,true_negative,false_positive,false_negative,true_positive,undirected_pair_count,directed_edge_count,note
0,GraphSAGE,0.844667,0.613556,0.263224,0.22726,0.591806,0.68,2427,259,207,107,1488139,2976278,"R-U-R + Product-Time 30 days, GraphSAGE baseline"


In [30]:
# 4. 모델 저장
model_path = graphsage_dir + '/itda_graphsage_model.pt'

torch.save({
    'model_state_dict': model.state_dict(),
    'feature_cols': feature_cols,
    'best_threshold': best_threshold,
    'best_valid_pr_auc': best_valid_pr_auc,
    'best_valid_macro_f1': best_valid_macro_f1,
    'edge_index_path': edge_index_path
}, model_path)

print("GraphSAGE model 저장 완료:", model_path)

GraphSAGE model 저장 완료: /content/drive/MyDrive/itda/baseline model_graphSAGE/itda_graphsage_model.pt
